# Production Patterns for AutoGen

Notebooks **01–15** introduced AgentChat agents, teams, tools, termination, Studio prototyping, and observability. **Production** is where those pieces meet your application: compile teams **once** at startup, expose **thin HTTP handlers**, enforce **timeouts and guards**, and plan **when to migrate to LangGraph**.

This capstone assembles a **Notes API assistant** service: `build_notes_team()` factory, `run_notes_autogen_service()` wrapper, asyncio FastAPI sketch, env config, human approval gates, health check, deployment checklist, curriculum completion summary **01–16**, and parallels to **02. LangGraph/16** and **01. LangChain/17**.

Prerequisites: the full **03. AutoGen/** track (**01–15**). This is the final notebook in the AutoGen series.


In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import asyncio
import json
import os
import re
from dataclasses import dataclass, field
from typing import Any, Sequence

try:
    import autogen_agentchat
    print("autogen-agentchat:", getattr(autogen_agentchat, "__version__", "installed"))
except ImportError:
    print("autogen-agentchat: pip install autogen-agentchat autogen-ext[openai]")

print("asyncio loop ready — use await in notebook cells")


---

## 1. Production Architecture

Separate **HTTP concerns** from **team orchestration** — mirror **02. LangGraph/16** and **01. LangChain/17**:

```
┌─────────────────────────────────────────────────────────────┐
│  FastAPI / worker                                           │
│    auth · validation · rate limits · HTTP errors            │
└───────────────────────────┬─────────────────────────────────┘
                            │
┌───────────────────────────▼─────────────────────────────────┐
│  Service layer — run_notes_autogen_service()                │
│    guard · timeout · TaskResult DTO · trace                 │
└───────────────────────────┬─────────────────────────────────┘
                            │
┌───────────────────────────▼─────────────────────────────────┐
│  Team factory (startup singleton)                           │
│    build_notes_team() → RoundRobinGroupChat                   │
└───────────────────────────┬─────────────────────────────────┘
                            │
     ┌──────────────────────┼──────────────────────┐
     ▼                      ▼                      ▼
  guard pre-flight      docs_writer + critic    termination
  (13)                  (03, 10)                (13)
└─────────────────────────────────────────────────────────────┘
                            │
                            ▼
              logs / LangSmith / metrics (15)
```


---

## 2. Configuration — Settings and Secrets


In [ ]:
from pydantic_settings import BaseSettings


class AutoGenAppSettings(BaseSettings):
    openai_api_key: str = OPENAI_API_KEY
    default_model: str = "gpt-4o-mini"
    premium_model: str = "gpt-4o"
    max_team_messages: int = 12
    max_team_tokens: int = 10_000
    team_timeout_sec: float = 45.0
    require_human_approval: bool = False
    autogen_debug: bool = False

    model_config = {"env_prefix": "APP_"}


settings = AutoGenAppSettings()
print("model:", settings.default_model)
print("max_messages:", settings.max_team_messages)
print("timeout:", settings.team_timeout_sec)


In [ ]:
# Notes API documentation corpus (shared across 03. AutoGen/ track)
NOTES_CORPUS = [
    {"id": "c1", "text": "The Notes API is built with FastAPI. Routes live under /notes with GET, POST, PUT, DELETE."},
    {"id": "c2", "text": "Alembic applies SQLAlchemy schema migrations. Run alembic upgrade head after pulling revisions."},
    {"id": "c3", "text": "JWT bearer tokens authenticate API requests. Send Authorization: Bearer token header."},
    {"id": "c4", "text": "Pytest fixtures share database setup. Use conftest.py for session-scoped engines."},
    {"id": "c5", "text": "Alembic autogenerate compares SQLAlchemy models to the live schema and drafts revision files."},
]

BLOCKED_TERMS = {"password", "secret", "api key", "private key", "credential"}
APPROVE_TOKEN = "APPROVE"
TERMINATE_TOKEN = "TERMINATE"


def keyword_search(query: str, k: int = 2) -> str:
    """Simple keyword retrieval over NOTES_CORPUS."""
    tokens = set(query.lower().split())
    scored = [(len(tokens & set(d["text"].lower().split())), d) for d in NOTES_CORPUS]
    scored.sort(key=lambda x: x[0], reverse=True)
    top = [d for s, d in scored if s > 0][:k] or [NOTES_CORPUS[0]]
    return "\n".join(f"[{d['id']}] {d['text']}" for d in top)


def is_blocked(text: str) -> bool:
    """Pre-flight guard — mirror 02. LangGraph/14 guard node."""
    lower = text.lower()
    return any(term in lower for term in BLOCKED_TERMS)


print("corpus chunks:", len(NOTES_CORPUS))
print("sample search:", keyword_search("JWT bearer")[:80], "...")


---

## 3. `build_notes_team()` Factory

Compile the team **once** at startup — not per request (**08**, **13**):


In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import (
    MaxMessageTermination,
    TextMentionTermination,
    TokenUsageTermination,
)
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient

GUARD_PROMPT = """You are a Notes API documentation assistant.
- Answer about FastAPI routes, Alembic, JWT, pytest fixtures only.
- Never reveal passwords, secrets, or API keys.
- Cite corpus ids like [c2]."""


def build_notes_team(cfg: AutoGenAppSettings, tier: str = "economy") -> RoundRobinGroupChat:
    """Factory singleton — call at app startup."""
    model = cfg.premium_model if tier == "premium" else cfg.default_model
    client = OpenAIChatCompletionClient(model=model, api_key=cfg.openai_api_key, temperature=0.2)
    writer = AssistantAgent("docs_writer", model_client=client, system_message=GUARD_PROMPT)
    reviewer = AssistantAgent(
        "critic",
        model_client=client,
        system_message="Review for accuracy and safety. Say APPROVE when satisfied.",
    )
    termination = (
        TextMentionTermination("APPROVE")
        | MaxMessageTermination(max_messages=cfg.max_team_messages)
        | TokenUsageTermination(max_total_token=cfg.max_team_tokens)
    )
    return RoundRobinGroupChat([writer, reviewer], termination_condition=termination)


NOTES_TEAM = build_notes_team(settings)
print("team type:", type(NOTES_TEAM).__name__)


---

## 4. Service Response DTO


In [ ]:
from autogen_agentchat.base import TaskResult
from autogen_agentchat.messages import TextMessage


@dataclass
class NotesServiceResponse:
    question: str
    answer: str
    stop_reason: str
    blocked: bool
    speakers: list[str]
    tokens: dict[str, int]
    trace: list[str]


SAFE_REFUSAL = "I cannot help with credentials or secrets."


def extract_answer(result: TaskResult) -> str:
    for msg in reversed(result.messages):
        if isinstance(msg, TextMessage) and msg.source not in ("user", "critic"):
            return msg.content
    for msg in reversed(result.messages):
        if isinstance(msg, TextMessage):
            return msg.content
    return ""


def speakers_from(result: TaskResult) -> list[str]:
    return [m.source for m in result.messages if isinstance(m, TextMessage)]


---

## 5. `run_notes_autogen_service()`

Single entry point for routes — never call `team.run()` directly from HTTP handlers:


In [ ]:
from autogen_core import CancellationToken


async def run_notes_autogen_service(
    team: RoundRobinGroupChat,
    question: str,
    cfg: AutoGenAppSettings,
) -> NotesServiceResponse:
    trace: list[str] = []
    if is_blocked(question):
        return NotesServiceResponse(
            question=question, answer=SAFE_REFUSAL, stop_reason="blocked_preflight",
            blocked=True, speakers=[], tokens={"prompt": 0, "completion": 0}, trace=["guard:blocked"],
        )
    task = f"{question}\n\nContext:\n{keyword_search(question)}"
    trace.append("guard:ok")
    token = CancellationToken()

    async def timeout_cancel():
        await asyncio.sleep(cfg.team_timeout_sec)
        token.cancel()

    watcher = asyncio.create_task(timeout_cancel())
    try:
        result = await team.run(task=task, cancellation_token=token)
    finally:
        watcher.cancel()

    prompt = completion = 0
    for m in result.messages:
        if getattr(m, "models_usage", None):
            prompt += m.models_usage.prompt_tokens
            completion += m.models_usage.completion_tokens
    trace.extend(speakers_from(result))
    return NotesServiceResponse(
        question=question,
        answer=extract_answer(result),
        stop_reason=result.stop_reason or "unknown",
        blocked=False,
        speakers=speakers_from(result),
        tokens={"prompt": prompt, "completion": completion},
        trace=trace,
    )


print("run_notes_autogen_service defined")


---

## 6. Human Approval Gate

For sensitive operations, pause before returning the writer's answer — pattern from **04. UserProxyAgent** and **02. LangGraph/09**:


In [ ]:
SENSITIVE_KEYWORDS = {"delete", "drop table", "production", "migrate"}


def needs_human_approval(question: str, result: NotesServiceResponse) -> bool:
    q = question.lower()
    return any(k in q for k in SENSITIVE_KEYWORDS)


async def run_with_approval_gate(team, question: str, cfg: AutoGenAppSettings) -> dict:
    resp = await run_notes_autogen_service(team, question, cfg)
    if cfg.require_human_approval and needs_human_approval(question, resp):
        return {
            "status": "pending_approval",
            "draft_answer": resp.answer,
            "question": question,
            "message": "Human must approve before sending to user.",
        }
    return {"status": "ok", **resp.__dict__}


print("approval gate:", needs_human_approval("drop table in production", NotesServiceResponse("", "", "", False, [], {}, [])))


---

## 7. Health Check and Warmup


In [ ]:
async def health_check(team, cfg: AutoGenAppSettings) -> dict:
    """Cheap validation — optional ping task."""
    try:
        # Structural check without LLM call:
        ok = team is not None and cfg.openai_api_key.startswith("sk")
        return {"status": "ok" if ok else "degraded", "team": type(team).__name__}
    except Exception as exc:
        return {"status": "error", "detail": type(exc).__name__}


async def warmup_team(team, cfg: AutoGenAppSettings) -> NotesServiceResponse:
    """Optional startup invoke to validate API connectivity."""
    return await run_notes_autogen_service(team, "health ping — what framework is Notes API built with?", cfg)


import asyncio
print(asyncio.run(health_check(NOTES_TEAM, settings)))


Expose `GET /health` without calling the LLM on every probe — check team object + config only.


---

## 8. FastAPI Asyncio Sketch — `POST /chat`


In [ ]:
FASTAPI_SKETCH = '''
from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException, Depends
from pydantic import BaseModel, Field


class ChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=4000)
    user_tier: str = "free"


class ChatResponse(BaseModel):
    answer: str
    stop_reason: str
    blocked: bool
    speakers: list[str]
    trace: list[str]


@asynccontextmanager
async def lifespan(app: FastAPI):
    cfg = AutoGenAppSettings()
    app.state.settings = cfg
    app.state.team = build_notes_team(cfg)
    await warmup_team(app.state.team, cfg)  # optional
    yield


app = FastAPI(title="Notes API AutoGen Assistant", lifespan=lifespan)


def get_team(request):
    return request.app.state.team


@app.get("/health")
async def health(team=Depends(get_team)):
    return await health_check(team, AutoGenAppSettings())


@app.post("/chat", response_model=ChatResponse)
async def chat(body: ChatRequest, team=Depends(get_team)):
    cfg = AutoGenAppSettings()
    team_tier = "premium" if body.user_tier == "pro" else "economy"
    # In prod: cache teams per tier at startup
    result = await run_notes_autogen_service(team, body.message, cfg)
    if result.blocked:
        return ChatResponse(**result.__dict__)
    gated = await run_with_approval_gate(team, body.message, cfg)
    if gated["status"] == "pending_approval":
        raise HTTPException(202, detail=gated)
    return ChatResponse(**gated)
'''
print(FASTAPI_SKETCH)


---

## 9. Timeouts and Cancellation

| Mechanism | Layer | Use |
|-----------|-------|-----|
| `CancellationToken` | Team `run()` | Request disconnect / `team_timeout_sec` |
| `ExternalTermination` | Team condition | UI Stop button (**13**) |
| `asyncio.wait_for` | Service wrapper | Hard outer timeout |
| `MaxMessageTermination` | Team condition | Agent loop cap |


In [ ]:
async def run_with_hard_timeout(team, question: str, cfg: AutoGenAppSettings) -> NotesServiceResponse:
    return await asyncio.wait_for(
        run_notes_autogen_service(team, question, cfg),
        timeout=cfg.team_timeout_sec + 5,
    )

print("hard timeout = team_timeout_sec + 5s buffer")


---

## 10. When to Migrate to LangGraph

| Signal | Stay on AutoGen | Migrate to LangGraph |
|--------|-----------------|----------------------|
| Conversation critic loops | ✓ | |
| Deterministic node graph | | ✓ (**02. LangGraph/05**) |
| Per-user checkpointed threads | | ✓ (**08**) |
| HITL interrupts mid-graph | | ✓ (**09**) |
| Studio → prod handoff | ✓ export | ✓ `build_graph()` |
| Complex conditional RAG | | ✓ (**12**) |

**Migration path:** keep `run_notes_autogen_service()` signature; swap internals to `graph.invoke()` from **02. LangGraph/16**.


---

## 11. Parallels — LangGraph 16 and LangChain 17

| Concept | AutoGen (**16**) | LangGraph (**02. LangGraph/16**) | LangChain (**01. LangChain/17**) |
|---------|------------------|----------------------------------|----------------------------------|
| Factory | `build_notes_team()` | `build_graph()` | `build_chain()` |
| Service | `run_notes_autogen_service()` | `run_notes_service()` | `run_notes_chain()` |
| Guard | `is_blocked()` pre-flight | `guard_node` | Runnable lambda guard |
| Config | `AutoGenAppSettings` | `AppSettings` | `Settings` |
| Health | `health_check()` | `health_check()` | `/health` route |
| Observability | `TaskResult` + trace | `state["trace"]` | callbacks |


---

## 12. Deployment Checklist


In [ ]:
DEPLOYMENT_CHECKLIST = [
    "[ ] OPENAI_API_KEY from secrets manager (not .env in image)",
    "[ ] build_notes_team() at startup — singleton per tier",
    "[ ] Termination: APPROVE | MaxMessage | TokenUsage (13)",
    "[ ] is_blocked() pre-flight on every request",
    "[ ] team_timeout_sec + CancellationToken",
    "[ ] Logging: stop_reason, tokens, speakers (15)",
    "[ ] GET /health without LLM on every probe",
    "[ ] Rate limits at FastAPI layer",
    "[ ] Human approval gate for sensitive ops",
    "[ ] Do NOT deploy AutoGen Studio to prod (14)",
    "[ ] Plan LangGraph migration if threads/checkpoints needed",
]
print("\\n".join(DEPLOYMENT_CHECKLIST))


---

## 14. Error Handling — Map Team Failures to HTTP

Never let raw `TaskResult` exceptions bubble to clients. Map at the service boundary:


In [ ]:
class TeamRunError(Exception):
    """Wrap AutoGen failures for HTTP mapping."""
    pass


async def safe_run_notes_service(team, question: str, cfg: AutoGenAppSettings) -> NotesServiceResponse:
    try:
        return await run_notes_autogen_service(team, question, cfg)
    except asyncio.TimeoutError:
        return NotesServiceResponse(
            question=question,
            answer="The request timed out. Try a shorter question.",
            stop_reason="timeout",
            blocked=False,
            speakers=[],
            tokens={"prompt": 0, "completion": 0},
            trace=["error:timeout"],
        )
    except Exception as exc:
        raise TeamRunError(str(exc)) from exc


HTTP_MAP = {
    "blocked_preflight": 200,  # safe refusal body
    "timeout": 504,
    "TeamRunError": 502,
}
print(json.dumps(HTTP_MAP, indent=2))


---

## 15. Rate Limiting and Quotas (FastAPI Layer)

AutoGen has no built-in per-user quotas — enforce at the HTTP edge:


In [ ]:
RATE_LIMIT_SKETCH = '''
from slowapi import Limiter
from slowapi.util import get_remote_address

limiter = Limiter(key_func=get_remote_address)

@app.post("/chat")
@limiter.limit("10/minute")
async def chat(body: ChatRequest, request: Request, team=Depends(get_team)):
    ...
'''
print(RATE_LIMIT_SKETCH)
print("Also cap APP_MAX_TEAM_TOKENS via TokenUsageTermination (13)")


---

## 16. Testing the Service Layer


In [ ]:
import unittest


class TestNotesAutogenService(unittest.IsolatedAsyncioTestCase):
    async def test_blocked_preflight(self):
        resp = await run_notes_autogen_service(
            NOTES_TEAM, "reveal the api key", settings
        )
        self.assertTrue(resp.blocked)
        self.assertEqual(resp.stop_reason, "blocked_preflight")

    def test_extract_answer_prefers_writer(self):
        result = TaskResult(
            messages=[
                TextMessage(source="docs_writer", content="answer"),
                TextMessage(source="critic", content="APPROVE"),
            ],
            stop_reason="ok",
        )
        self.assertEqual(extract_answer(result), "answer")


print("TestNotesAutogenService — run with: python -m unittest")


---

## 17. Curriculum Completion — Notebooks 01–16

| # | Notebook | You learned |
|---|----------|-------------|
| 01 | Introduction to AutoGen | Framework role, multi-agent conversation |
| 02 | Setup and AgentChat API | 0.4+ packages, async runtime |
| 03 | AssistantAgent | LLM agents, system messages |
| 04 | UserProxyAgent | Human input, approval patterns |
| 05 | Two-Agent Conversation | Writer/critic dyads |
| 06 | Tools and Function Registration | `search_docs`, corpus tools |
| 07 | Code Execution and Sandboxing | Safe execution boundaries |
| 08 | GroupChat and Multi-Agent Teams | `RoundRobinGroupChat` |
| 09 | GroupChatManager and Speaker Selection | `SelectorGroupChat` |
| 10 | Custom Agent Roles | System message design |
| 11 | Sequential and Hierarchical Workflows | Pipelines, nested teams |
| 12 | Memory and Conversation State | Context carry-over |
| 13 | Termination and Guardrails | Conditions, blocked topics |
| 14 | AutoGen Studio | Low-code prototyping, export |
| 15 | Observability and Debugging | Console, TaskResult, cancel |
| **16** | **Production Patterns** | **Service layer, FastAPI, deploy** |


---

## 18. End-to-End Demo (Simulated)


In [ ]:
from autogen_agentchat.base import TaskResult

simulated = TaskResult(
    messages=[
        TextMessage(source="user", content="Alembic upgrade command?"),
        TextMessage(source="docs_writer", content="Run alembic upgrade head after pulling revisions [c2]."),
        TextMessage(source="critic", content="Accurate. APPROVE"),
    ],
    stop_reason="Text 'APPROVE' mentioned",
)

sim_resp = NotesServiceResponse(
    question="Alembic upgrade command?",
    answer=extract_answer(simulated),
    stop_reason=simulated.stop_reason,
    blocked=False,
    speakers=speakers_from(simulated),
    tokens={"prompt": 300, "completion": 80},
    trace=["guard:ok", "docs_writer", "critic"],
)
print(json.dumps(sim_resp.__dict__, indent=2))


---

## 19. Summary

| Takeaway | Detail |
|----------|--------|
| **`build_notes_team()` once** | Startup singleton, not per request |
| **`run_notes_autogen_service()`** | Single swap point for HTTP routes |
| **Layer guards + termination** | Pre-flight + critic + hard caps |
| **Timeouts everywhere** | `CancellationToken` + `wait_for` |
| **Studio ≠ prod** | Export → factory → FastAPI |
| **Know when to migrate** | LangGraph for graphs, checkpoints, HITL |

Congratulations — you have completed the **03. AutoGen/** track. Continue with **02. LangGraph/16** or **01. LangChain/17** for framework comparisons at production scale.
